In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
import re
import numpy as np
import sys 

# Path to Environment resources
env_path = f"{notebookutils.nbResPath}/env"

if env_path not in sys.path:
    sys.path.append(env_path)

from udf_sharepoint import download_file_from_sharepoint, upload_file_2_sharepoint # download_file_from_sharepoint(sp_folder_name, file_name, lakehouse_folder_path) ,  upload_file_2_sharepoint(full_path_2_filename,sp_folder_name)
from data_cleaning_utils import auto_cast_string_columns, clean_columns


StatementMeta(, f5efd814-357d-4e4a-87e1-b955bae31297, 5, Finished, Available, Finished, False)

In [2]:

# ── Load the table ───────────────────────────────────────────────
df_scorecard = spark.table("srm.prophet_scorecard").toPandas()

# ── Update this one line each month ────────────────────────────
CURRENT_MONTH = pd.Timestamp("2026-07-01")

# ── Add Current_Month column in YYYYMM format ──────────────────
df_scorecard["Current_Month"] = CURRENT_MONTH.strftime("%Y%m")

# ── Normalize any stray "nan"/"None"/"NaT" TEXT to real NaN first ─
def normalize_missing(val):
    if isinstance(val, str) and val.strip().lower() in ("nan", "none", "nat", ""):
        return np.nan
    return val

# ── Strip "(Watch)"/"(Low)"/etc. text from period columns, keep just the number ─
def extract_numeric(val):
    if pd.isna(val):
        return val
    if isinstance(val, str):
        match = re.match(r'^\s*(-?\d+\.?\d*)', val)
        if match:
            return float(match.group(1))
    return val

for col in ["Current", "M1", "M2", "M3"]:
    df_scorecard[col] = df_scorecard[col].apply(extract_numeric)

df_scorecard = df_scorecard.applymap(normalize_missing)

# ── Replace NaN with blank for a clean Excel export ─────────────
df_scorecard = df_scorecard.fillna("")


# ── Output path — Lakehouse Files section, downloadable from there ─
output_path = "/lakehouse/default/Files/prophet_scorecard_export.xlsx"

commodities = ["Wheat", "Corn", "Rice", "Soybean", "Barley"]


StatementMeta(, f5efd814-357d-4e4a-87e1-b955bae31297, 6, Finished, Available, Finished, False)

/tmp/ipykernel_12868/1710382254.py:29: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_scorecard = df_scorecard.applymap(normalize_missing)


In [3]:
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Sheet 1 — all records
    df_scorecard.to_excel(writer, sheet_name="All Commodities", index=False)

    # Sheets 2-6 — one per commodity
    for commodity in commodities:
        df_c = df_scorecard[df_scorecard["Commodity"] == commodity]
        df_c.to_excel(writer, sheet_name=commodity, index=False)

print(f"✓ Excel file saved: {output_path}")
print(f"  Current_Month: {CURRENT_MONTH.strftime('%Y%m')}")
print(f"  Columns: {df_scorecard.columns.tolist()}")
print(f"  Sheet 'All Commodities': {len(df_scorecard)} rows")
for commodity in commodities:
    n = len(df_scorecard[df_scorecard["Commodity"] == commodity])
    print(f"  Sheet '{commodity}': {n} rows")

upload_file_2_sharepoint(output_path, "EWS_Datasets/Final_Model_Results")
print("Final Model Excel uploaded to SharePoint from", output_path)

StatementMeta(, f5efd814-357d-4e4a-87e1-b955bae31297, 7, Finished, Available, Finished, False)

✓ Excel file saved: /lakehouse/default/Files/prophet_scorecard_export.xlsx
  Current_Month: 202607
  Columns: ['Commodity', 'Domain', 'Variable', 'Direction', 'Weight_pct', 'Baseline', 'Current', 'M1', 'M2', 'M3', 'Score', 'Weighted_Avg', 'Low_Good', 'Watch', 'Warning', 'Emergency', 'Writeup_Current', 'Writeup_M1', 'Writeup_M2', 'Writeup_M3', 'Current_Month']
  Sheet 'All Commodities': 40 rows
  Sheet 'Wheat': 8 rows
  Sheet 'Corn': 8 rows
  Sheet 'Rice': 8 rows
  Sheet 'Soybean': 8 rows
  Sheet 'Barley': 8 rows
{'@odata.context': "https://graph.microsoft.com/v1.0/$metadata#sites('41577faa-cc80-40ec-9cd9-e78e9d25b512%2C410ab16c-260a-42aa-b175-e60887a1705c')/drives('b%21qn9XQYDM7ECc2eeOnSW1EmyxCkEKJqpCsXXmCIehcFyZKrUC4Xq6Qrfw8isB_dqp')/items/$entity", '@microsoft.graph.downloadUrl': 'https://salic.sharepoint.com/sites/ResearchPipeline/_layouts/15/download.aspx?UniqueId=c692f777-ebf7-4d91-93a4-47e7ab5d9229&Translate=false&tempauth=v1.eyJzaXRlaWQiOiI0MTU3N2ZhYS1jYzgwLTQwZWMtOWNkOS1lNzhlOW

In [4]:
# with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
#     # Sheet 1 — all records
#     df_scorecard.to_excel(writer, sheet_name="All Commodities", index=False)

#     # Sheets 2-6 — one per commodity
#     for commodity in commodities:
#         df_c = df_scorecard[df_scorecard["Commodity"] == commodity]
#         df_c.to_excel(writer, sheet_name=commodity, index=False)

# print(f"✓ Excel file saved: {output_path}")
# print(f"  Current_Month: {CURRENT_MONTH.strftime('%Y%m')}")
# print(f"  Sheet 'All Commodities': {len(df_scorecard)} rows")
# for commodity in commodities:
#     n = len(df_scorecard[df_scorecard["Commodity"] == commodity])
#     print(f"  Sheet '{commodity}': {n} rows")

StatementMeta(, f5efd814-357d-4e4a-87e1-b955bae31297, 8, Finished, Available, Finished, False)